# Convolutional Neutral Network (CNN)

In [12]:
import os
import glob
import numpy as np 
import pandas as pd 
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                            classification_report, confusion_matrix)

torch.manual_seed(seed=42)
np.random.seed(seed=42)

## <font color="orange">Load MFCCs and specify labels and groups </font>

In [2]:
# emotion
EMO = {"W":"anger","L":"boredom",
    "E":"disgust","A":"fear",
    "F":"happiness","T":"sadness","N":"neutral"}

# read all all mfcc csv files
paths = sorted(glob.glob("../data/extraction/mfcc_mat/*.csv")) 
mfccs, labels, groups = [], [], []
for p in paths:
    name = os.path.splitext(os.path.basename(p))[0]  # e.g. 03a01Fa
    m = pd.read_csv(p, header=None).values.astype(np.float32) #(n_frames, n_mfcc)
    mfccs.append(m)
    groups.append(name[:2])           # speaker id
    labels.append(EMO[name[5]])       # emotion letter

# encode the target
le = LabelEncoder()
y = le.fit_transform(labels)
groups = np.array(groups)
print(len(mfccs), le.classes_, np.unique(groups))

567 ['anger' 'boredom' 'disgust' 'fear' 'happiness' 'neutral' 'sadness'] ['03' '08' '09' '10' '11' '12' '13' '14' '15' '16']


## <font color="orange"> Preparation </font>

In [3]:
def fix_len(m, T):
    """Crop or zero-pad along frames dim to exactly T frames."""
    if m.shape[0] >= T:
        return m[:T]
    return np.pad(m, ((0, T - m.shape[0]), (0, 0)))

In [ ]:
# choose the test and validation groups
test_spk = ["09", "14"] # 1 male + 1 female
val_spk  = ["10", "15"] # 1 male + 1 female

# filter machines
is_test = np.isin(groups, test_spk)
is_val  = np.isin(groups, val_spk)
is_tr   = ~(is_test | is_val)

# Set T
T = int(np.percentile([m.shape[0] for m, t in zip(mfccs, is_tr) if t], 95))
print(f"T = {T}")

# mean/std over all unpadded/uncropped train frames for each mfcc
# (only train)
tr_frames = np.concatenate([m for m, t in zip(mfccs, is_tr) if t], axis=0)
mean, std = tr_frames.mean(0), tr_frames.std(0) + 1e-8
# Normalize first, then crop/pad
X = np.stack([fix_len((m - mean) / std, T) for m in mfccs]).astype(np.float32)

# split
split = {}
for name, mask in [("train", is_tr), ("val", is_val), ("test", is_test)]:
    split[name] = (X[mask], y[mask])

(Xtr, ytr) = split["train"] 
(Xva, yva) = split["val"]
(Xte, yte) = split["test"]

print("Train/test/val split ok!")

T = 503
Train/test/val split ok!


In [7]:
# compute class weights
n_classes = len(le.classes_)
counts = np.bincount(ytr, minlength=n_classes)
class_weights = counts.sum() / (n_classes * np.maximum(counts, 1))
print(le.classes_)
print(class_weights)

['anger' 'boredom' 'disgust' 'fear' 'happiness' 'neutral' 'sadness']
[0.63079777 0.88311688 2.02380952 1.21428571 1.01190476 0.97142857
 1.05590062]


In [8]:
# augmentation (use in train)
# masking + small gaussian noise
def augment(x, max_t=15, max_f=5, noise=0.05):
    """x is batch of shape (B,T,F): batch size, time frames, MFCC coeffs"""
    x = x.clone() # keep original batch safe
    B, T, F = x.shape
    for i in range(B):
        # time frame mask
        t = np.random.randint(0, max_t+1) # random mask width
        t0 = np.random.randint(0, max(T-t,1)) # random start position of the mask
        x[i, t0:t0+t, :] = 0 # masking with 0 -> nrutral info
        
        #frequency mask (on the MFCC axis)
        f = np.random.randint(0, max_f+1)
        f0 = np.random.randint(0, max(F-f,1))
        x[i, :, f0:f0+f] = 0
    # gaussian noise
    xn = x + noise * torch.rand_like(x)
    return xn

In [11]:
# loader
def make_loader(X, y, shuffle, batch=32):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

## <font color="orange"> CNN Construction </font>

### **1D CNN**

In [9]:
# 1D convolutional block
def conv_block_1d(cin, cout):
    return nn.Sequential(
        # filters along time axis, each filter looks at 5 consecutive frames 
        #-> padding=2
        nn.Conv1d(cin, cout, kernel_size=5, padding=2),
        nn.BatchNorm1d(cout),
        nn.ReLU(),
        nn.MaxPool1d(2),
    )

In [ ]:
# convolves over time 
# MFCC are channels
class MFCC_Net_1D(nn.Module):
    def __init__(self, n_mfcc, n_classes, dropout=0.4):
        super().__init__()
        # learns pattern over time
        self.features = nn.Sequential(
            conv_block_1d(n_mfcc, 64), 
            conv_block_1d(64, 128),
            conv_block_1d(128,128),
            nn.AdaptiveAvgPool1d(1) # global average and leaving 1 per channel
        )
        # fully connected layers turning pattern into emotion scores
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), # a hidden FC layer
            nn.ReLU(), # add nonlinearity
            nn.Dropout(dropout), # reduce overfitting by randomly zeroing dropout ratio of 64 values during training
            nn.Linear(64, n_classes), # output 1 raw score per emotion
        )
    def forward(self, x):
        return self.classifier(self.features(x.transpose(1,2)))


## **2D CNN**

In [ ]:
def conv_block_2d(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel_size=3, padding=1),
        nn.BatchNorm2d(cout),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

In [ ]:
# CNN 2D: treat (T, n_mfcc) as a 1-channel image
class MFCCNet2D(nn.Module):
    def __init__(self, n_classes, dropout=0.3):
        super().__init__()
        # features extraction
        self.features = nn.Sequential(
            conv_block_2d(1,32),
            conv_block_2d(32,64),
            conv_block_2d(64,128),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):  # x: (B, T, n_mfcc)
        return self.classifier(self.features(x.unsqueeze(1))) 

## <font color="orange"> Training </font>